In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
pd.set_option('display.max_columns', None)
print('Successfully Loaded!')


Successfully Loaded!


In [8]:
import os
import numpy as np
import pandas as pd

# =========================
# 1. 路径设置
# =========================
# 改成你的 csv 所在文件夹
BASE_PATH = "."

# =========================
# 2. 工具函数
# =========================
def mode_or_nan(series):
    """返回众数；如果全空则返回 NaN"""
    series = series.dropna()
    if series.empty:
        return np.nan
    m = series.mode()
    return m.iloc[0] if not m.empty else series.iloc[0]

def haversine_km(lat1, lon1, lat2, lon2):
    """
    计算两点经纬度球面距离（单位：km）
    支持 pandas Series
    """
    lat1 = np.radians(pd.to_numeric(lat1, errors="coerce"))
    lon1 = np.radians(pd.to_numeric(lon1, errors="coerce"))
    lat2 = np.radians(pd.to_numeric(lat2, errors="coerce"))
    lon2 = np.radians(pd.to_numeric(lon2, errors="coerce"))

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )
    c = 2 * np.arcsin(np.sqrt(a))
    return 6371 * c  # 地球半径约 6371 km

# =========================
# 3. 读取数据
# =========================
from pathlib import Path
BASE_PATH = Path("Sprintinternship_Workshop") / "4_DeliveryDelayPrediction" / "csv_files"

print("当前工作目录:", Path.cwd())
print("数据目录是否存在:", BASE_PATH.exists())
print("orders 文件是否存在:", (BASE_PATH / "olist_orders_dataset.csv").exists())

orders = pd.read_csv(
    "./csv_files/olist_orders_dataset.csv",
    parse_dates=[
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ],
)

customers = pd.read_csv("./csv_files/olist_customers_dataset.csv")

items = pd.read_csv(
    "./csv_files/olist_order_items_dataset.csv",
    parse_dates=["shipping_limit_date"],
)

products = pd.read_csv("./csv_files/olist_products_dataset.csv")
sellers = pd.read_csv("./csv_files/olist_sellers_dataset.csv")
geo = pd.read_csv("./csv_files/olist_geolocation_dataset.csv")
payments = pd.read_csv("./csv_files/olist_order_payments_dataset.csv")
translation = pd.read_csv("./csv_files/product_category_name_translation.csv")

# 注意：
# reviews 表这里故意不接进来，因为评论和评分发生在送达之后，
# 用它来预测是否延误会造成数据泄漏。

# =========================
# 4. 处理 geolocation
# =========================
# geolocation 同一个 zip_code_prefix 有很多行，先聚合成每个 zip 前缀一行
geo_agg = (
    geo.groupby("geolocation_zip_code_prefix", as_index=False)
    .agg(
        geo_lat=("geolocation_lat", "mean"),
        geo_lng=("geolocation_lng", "mean"),
        geo_city=("geolocation_city", mode_or_nan),
        geo_state=("geolocation_state", mode_or_nan),
    )
)

# 给 customer 用
customer_geo = geo_agg.rename(
    columns={
        "geolocation_zip_code_prefix": "customer_zip_code_prefix",
        "geo_lat": "customer_lat",
        "geo_lng": "customer_lng",
        "geo_city": "customer_geo_city",
        "geo_state": "customer_geo_state",
    }
)

# 给 seller 用
seller_geo = geo_agg.rename(
    columns={
        "geolocation_zip_code_prefix": "seller_zip_code_prefix",
        "geo_lat": "seller_lat",
        "geo_lng": "seller_lng",
        "geo_city": "seller_geo_city",
        "geo_state": "seller_geo_state",
    }
)

# =========================
# 5. 处理 products
# =========================
# 类别翻译
products = products.merge(translation, on="product_category_name", how="left")

# 构造体积特征
products["product_volume_cm3"] = (
    products["product_length_cm"]
    * products["product_height_cm"]
    * products["product_width_cm"]
)

# =========================
# 6. 处理 sellers
# =========================
sellers = sellers.merge(seller_geo, on="seller_zip_code_prefix", how="left")

# =========================
# 7. 构造 item-level 明细，再聚合成 order-level
# =========================
item_level = (
    items.merge(products, on="product_id", how="left")
        .merge(sellers, on="seller_id", how="left")
)

order_item_features = (
    item_level.groupby("order_id", as_index=False)
    .agg(
        item_count=("order_item_id", "count"),
        unique_product_count=("product_id", "nunique"),
        seller_count=("seller_id", "nunique"),

        total_price=("price", "sum"),
        avg_price=("price", "mean"),
        max_price=("price", "max"),

        total_freight=("freight_value", "sum"),
        avg_freight=("freight_value", "mean"),

        total_weight_g=("product_weight_g", "sum"),
        avg_weight_g=("product_weight_g", "mean"),

        total_volume_cm3=("product_volume_cm3", "sum"),
        avg_volume_cm3=("product_volume_cm3", "mean"),

        avg_photos_qty=("product_photos_qty", "mean"),

        main_category=("product_category_name_english", mode_or_nan),
        main_seller_state=("seller_state", mode_or_nan),
        seller_state_count=("seller_state", "nunique"),

        # 多卖家订单这里取卖家经纬度平均值，作为订单整体的 seller 位置近似
        seller_lat_mean=("seller_lat", "mean"),
        seller_lng_mean=("seller_lng", "mean"),
    )
)

# =========================
# 8. 处理 payments
# =========================
payments["is_credit_card"] = (payments["payment_type"] == "credit_card").astype(int)

payment_features = (
    payments.groupby("order_id", as_index=False)
    .agg(
        payment_value_total=("payment_value", "sum"),
        payment_value_mean=("payment_value", "mean"),
        payment_installments_max=("payment_installments", "max"),
        payment_type_main=("payment_type", mode_or_nan),
        payment_type_count=("payment_type", "nunique"),
        is_credit_card_any=("is_credit_card", "max"),
    )
)

# =========================
# 9. 合并成总表（订单级）
# =========================
master = (
    orders.merge(customers, on="customer_id", how="left")
          .merge(customer_geo, on="customer_zip_code_prefix", how="left")
          .merge(order_item_features, on="order_id", how="left")
          .merge(payment_features, on="order_id", how="left")
)

# =========================
# 10. 构造关键特征
# =========================

# 客户-卖家距离
master["distance_km"] = haversine_km(
    master["seller_lat_mean"],
    master["seller_lng_mean"],
    master["customer_lat"],
    master["customer_lng"],
)

# 时间特征
master["approval_delay_hours"] = (
    master["order_approved_at"] - master["order_purchase_timestamp"]
).dt.total_seconds() / 3600

master["carrier_lead_hours"] = (
    master["order_delivered_carrier_date"] - master["order_approved_at"]
).dt.total_seconds() / 3600

master["estimated_delivery_days"] = (
    master["order_estimated_delivery_date"] - master["order_purchase_timestamp"]
).dt.total_seconds() / (3600 * 24)

master["actual_delivery_days"] = (
    master["order_delivered_customer_date"] - master["order_purchase_timestamp"]
).dt.total_seconds() / (3600 * 24)

master["delivery_delay_days"] = (
    master["order_delivered_customer_date"] - master["order_estimated_delivery_date"]
).dt.total_seconds() / (3600 * 24)

# 标签：是否延误
master["is_late"] = (master["delivery_delay_days"] > 0).astype("Int64")

# 下单时间拆分
master["purchase_hour"] = master["order_purchase_timestamp"].dt.hour
master["purchase_weekday"] = master["order_purchase_timestamp"].dt.weekday
master["purchase_month"] = master["order_purchase_timestamp"].dt.month
master["purchase_day"] = master["order_purchase_timestamp"].dt.day
master["is_weekend_purchase"] = master["purchase_weekday"].isin([5, 6]).astype(int)

# 是否跨州
master["cross_state"] = (
    master["customer_state"] != master["main_seller_state"]
).astype("Int64")

# =========================
# 11. 调整列顺序（更整洁）
# =========================
ordered_cols = [
    # 主键与订单基础信息
    "order_id", "customer_id", "customer_unique_id", "order_status",
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date",

    # 客户信息
    "customer_zip_code_prefix", "customer_city", "customer_state",
    "customer_lat", "customer_lng",
    "customer_geo_city", "customer_geo_state",

    # 订单聚合特征
    "item_count", "unique_product_count", "seller_count",
    "total_price", "avg_price", "max_price",
    "total_freight", "avg_freight",
    "total_weight_g", "avg_weight_g",
    "total_volume_cm3", "avg_volume_cm3",
    "avg_photos_qty",

    # 类别 / 卖家信息
    "main_category", "main_seller_state", "seller_state_count",
    "seller_lat_mean", "seller_lng_mean",

    # 支付信息
    "payment_value_total", "payment_value_mean",
    "payment_installments_max", "payment_type_main",
    "payment_type_count", "is_credit_card_any",

    # 工程特征
    "distance_km",
    "approval_delay_hours", "carrier_lead_hours",
    "estimated_delivery_days", "actual_delivery_days",
    "delivery_delay_days",
    "purchase_hour", "purchase_weekday", "purchase_month", "purchase_day",
    "is_weekend_purchase", "cross_state",

    # 标签
    "is_late",
]

master = master[ordered_cols]

# =========================
# 12. 生成“更适合直接建模”的版本
# =========================
# 只保留能形成标签的 delivered 订单
model_ready = master[
    (master["order_status"] == "delivered")
    & (master["order_delivered_customer_date"].notna())
    & (master["order_estimated_delivery_date"].notna())
].copy()

# 删除明显泄漏未来信息的列
# 这些列不能直接作为“提前预测是否延误”的输入特征
drop_cols_for_model = [
    "customer_id",
    "customer_unique_id",
    "order_delivered_customer_date",   # 真实送达时间，结果本身
    "order_delivered_carrier_date",    # 如果你要在下单时预测，这个时点还未知
    "actual_delivery_days",            # 结果本身
    "carrier_lead_hours",              # 依赖交给承运商时间
    "delivery_delay_days",             # 直接由标签定义出来
]

model_ready = model_ready.drop(columns=drop_cols_for_model)

# =========================
# 13. 导出文件
# =========================
master.to_csv(
    os.path.join("./csv_files/olist_order_master_clean.csv"),
    index=False,
    encoding="utf-8-sig"
)

model_ready.to_csv(
    os.path.join("./csv_files/olist_model_ready_late_delivery.csv"),
    index=False,
    encoding="utf-8-sig"
)

# =========================
# 14. 简单检查
# =========================
print("master shape:", master.shape)
print("model_ready shape:", model_ready.shape)
print("\n延误标签分布：")
print(model_ready["is_late"].value_counts(dropna=False))

print("\nmaster 前5行：")
print(master.head())

print("\nmodel_ready 前5行：")
print(model_ready.head())

当前工作目录: c:\Users\Lingtong\Desktop\Avd DSAI workshop\Sprinternship_Workshop\4_DeliveryDelayPrediciton
数据目录是否存在: False
orders 文件是否存在: False
master shape: (99441, 53)
model_ready shape: (96470, 46)

延误标签分布：
is_late
0    88644
1     7826
Name: count, dtype: Int64

master 前5行：
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

                 customer_unique_id order_status order_purchase_timestamp  \
0  7c396fd4830fd04220f754e42b4e5bff    delivered      2017-10-02 10:56:33   
1  af07308b275d755c9edb36a90c618231    delivered      2018-07-24 20:41:37   
2  3a653a41f6f9fc3d2a113cf8398680e8    delivered      201

In [2]:
import numpy as np
import pandas as pd

from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
)

# =========================
# 1) 读取训练表
# 如果你已经有 model_ready 这个 DataFrame，
# 就把下面这行改成: df = model_ready.copy()
# =========================
df = pd.read_csv("./csv_files/olist_model_ready_late_delivery.csv")

# =========================
# 2) 目标变量
# =========================
target_col = "is_late"
df = df[df[target_col].notna()].copy()
df[target_col] = df[target_col].astype(int)

# =========================
# 3) 删除不该进模型的列
# =========================
drop_cols = [
    target_col,
    "order_id",                    # 纯ID
    "order_status",                # delivered 后通常变成常量
    "order_purchase_timestamp",    # 原始时间戳字符串，先不用
    "order_approved_at",
    "order_estimated_delivery_date",
    "customer_geo_city",           # 与 customer_city 高度重复，可先删掉
    "customer_geo_state",
]

X = df.drop(columns=[c for c in drop_cols if c in df.columns]).copy()
y = df[target_col].copy()

# 删除常量列
nunique = X.nunique(dropna=False)
constant_cols = nunique[nunique <= 1].index.tolist()
if constant_cols:
    X = X.drop(columns=constant_cols)

# =========================
# 4) 识别类别列 / 数值列
# =========================
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

# CatBoost 的分类列不要保留 NaN / float 表示，统一转字符串
for c in cat_cols:
    X[c] = X[c].astype("string").fillna("__MISSING__").astype(str)

# 数值列尽量转成 numeric
for c in num_cols:
    X[c] = pd.to_numeric(X[c], errors="coerce")

print("X shape:", X.shape)
print("类别列数量:", len(cat_cols))
print("数值列数量:", len(num_cols))
print("延误比例:", y.mean())

# =========================
# 5) 分层切分：70% train / 15% valid / 15% test
# =========================
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y,
    test_size=0.15,
    random_state=42,
    stratify=y
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_trainval, y_trainval,
    test_size=0.17647,   # 0.85 * 0.17647 ≈ 0.15
    random_state=42,
    stratify=y_trainval
)

print("Train:", X_train.shape, y_train.shape)
print("Valid:", X_valid.shape, y_valid.shape)
print("Test :", X_test.shape, y_test.shape)

# =========================
# 6) 训练 CatBoost
# =========================
model = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.03,
    depth=8,
    loss_function="Logloss",
    eval_metric="AUC",
    auto_class_weights="Balanced",
    random_state=42,
    verbose=100
)

model.fit(
    X_train,
    y_train,
    cat_features=cat_cols,
    eval_set=(X_valid, y_valid),
    use_best_model=True,
    early_stopping_rounds=100
)

# =========================
# 7) 在验证集上调阈值（按 F1）
# =========================
valid_proba = model.predict_proba(X_valid)[:, 1]

best_threshold = 0.5
best_f1 = -1

for t in np.arange(0.10, 0.91, 0.01):
    pred_t = (valid_proba >= t).astype(int)
    f1_t = f1_score(y_valid, pred_t, zero_division=0)
    if f1_t > best_f1:
        best_f1 = f1_t
        best_threshold = float(t)

print(f"\nBest threshold on valid = {best_threshold:.2f}")
print(f"Best valid F1           = {best_f1:.4f}")

# =========================
# 8) 测试集评估
# =========================
test_proba = model.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= best_threshold).astype(int)

print("\n=== Test Metrics ===")
print("Accuracy :", round(accuracy_score(y_test, test_pred), 4))
print("Precision:", round(precision_score(y_test, test_pred, zero_division=0), 4))
print("Recall   :", round(recall_score(y_test, test_pred, zero_division=0), 4))
print("F1       :", round(f1_score(y_test, test_pred, zero_division=0), 4))
print("ROC-AUC  :", round(roc_auc_score(y_test, test_proba), 4))
print("PR-AUC   :", round(average_precision_score(y_test, test_proba), 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, test_pred))

print("\nClassification Report:")
print(classification_report(y_test, test_pred, digits=4, zero_division=0))

# =========================
# 9) 特征重要性
# =========================
fi = pd.DataFrame({
    "feature": X_train.columns,
    "importance": model.get_feature_importance()
}).sort_values("importance", ascending=False)

print("\nTop 20 Feature Importance:")
print(fi.head(20))

# =========================
# 10) 保存结果
# =========================
fi.to_csv("catboost_feature_importance.csv", index=False, encoding="utf-8-sig")
model.save_model("catboost_late_delivery.cbm")

print("\n已保存:")
print("- catboost_feature_importance.csv")
print("- catboost_late_delivery.cbm")

ModuleNotFoundError: No module named 'sklearn'